In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
file_path = r"D:\clv\data\raw\online_retail_II.xlsx"

all_sheets = pd.read_excel(file_path, sheet_name=None)

df = pd.concat(all_sheets.values(), ignore_index=True)

print(df.shape)

df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
clean_df = df.copy()

In [6]:
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

clean_df.columns

Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer_id', 'country'],
      dtype='str')

In [7]:
clean_df["invoice"] = clean_df["invoice"].astype(str)

clean_df["stockcode"] = clean_df["stockcode"].astype(str)

clean_df["invoicedate"] = pd.to_datetime(clean_df["invoicedate"])

clean_df["customer_id"] = pd.to_numeric(
    clean_df["customer_id"],
    errors="coerce"
)

In [8]:
print("Before:", clean_df.shape)

clean_df = clean_df.drop_duplicates().copy()

print("After:", clean_df.shape)

Before: (1067371, 8)
After: (1033036, 8)


In [9]:
print(clean_df["customer_id"].isna().sum())

clean_df = clean_df.dropna(
    subset=["customer_id"]
).copy()

clean_df["customer_id"] = clean_df["customer_id"].astype(int)

print(clean_df["customer_id"].isna().sum())

235151
0


In [10]:
clean_df["description"] = clean_df["description"].fillna(
    "Unknown Product"
)

In [11]:
for col in clean_df.select_dtypes(include="object"):

    clean_df[col] = clean_df[col].str.strip()

C:\Users\Urvi Patel\AppData\Local\Temp\ipykernel_12152\2129197972.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in clean_df.select_dtypes(include="object"):


In [12]:
clean_df["country"] = clean_df["country"].str.title()

clean_df["description"] = clean_df["description"].str.upper()

In [13]:
clean_df["is_cancelled"] = (
    clean_df["invoice"]
    .str.startswith("C")
)

clean_df["is_return"] = (
    clean_df["quantity"] < 0
)

In [17]:
transactions = clean_df.copy()
transactions["revenue"] = (
    transactions["quantity"] *
    transactions["price"]
)

transactions["year"] = transactions["invoicedate"].dt.year

transactions["month"] = transactions["invoicedate"].dt.month

transactions["month_name"] = (
    transactions["invoicedate"]
    .dt.month_name()
)

transactions["day"] = transactions["invoicedate"].dt.day

transactions["day_name"] = (
    transactions["invoicedate"]
    .dt.day_name()
)

transactions["hour"] = transactions["invoicedate"].dt.hour

transactions["quarter"] = (
    transactions["invoicedate"]
    .dt.quarter
)

transactions["week"] = (
    transactions["invoicedate"]
    .dt.isocalendar()
    .week
    .astype(int)
)

In [19]:
transactions.to_csv(
    "data/processed/clean_transactions.csv",
    index=False
)


print("Dataset saved successfully.")

Dataset saved successfully.
